# Lab: Simple & Multiple Linear Regression
**Exercise 1:** Simple Linear Regression from scratch (mtcars: mpg ~ weight)

**Exercise 2:** Multiple Linear Regression with feature selection (Boston housing: MEDV)

## Exercise 1

### 1. User-defined function `myLinReg(x, y)` for Simple Linear Regression
Given one predictor attribute `x` and one response attribute `y`, this function returns the slope and intercept of the best-fit straight line using the closed-form Ordinary Least Squares (OLS) formula:

$$ \hat{\beta}_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x} $$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def myLinReg(x, y):
    """
    Simple Linear Regression using the closed-form OLS solution.

    Parameters
    ----------
    x : array-like, shape (n_samples,)
        Predictor attribute.
    y : array-like, shape (n_samples,)
        Response attribute.

    Returns
    -------
    slope : float
        Coefficient (beta_1) of the fitted line.
    intercept : float
        Intercept (beta_0) of the fitted line.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_mean = np.mean(x)
    y_mean = np.mean(y)

    numerator = np.sum((x - x_mean) * (y - y_mean))
    denominator = np.sum((x - x_mean) ** 2)

    slope = numerator / denominator
    intercept = y_mean - slope * x_mean

    return slope, intercept


def myLinReg_predict(x, slope, intercept):
    """Predict response values given predictor x and fitted coefficients."""
    x = np.asarray(x, dtype=float)
    return intercept + slope * x


### 2. Load the `mtcars` dataset, split into train/test (80% / 20%)
`mtcars` is an R built-in dataset. We load it here from a public CSV mirror. If you don't have internet access on this machine, download `mtcars.csv` once and place it in the same folder as this notebook, then skip straight to `pd.read_csv("mtcars.csv")`.

In [ ]:
# Load mtcars (weight = 'wt' column, in units of 1000 lbs; mpg = 'mpg')
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/mtcars.csv"
try:
    mtcars = pd.read_csv(url)
except Exception as e:
    print("Could not fetch from URL, load mtcars.csv manually instead:", e)
    mtcars = pd.read_csv("mtcars.csv")

mtcars = mtcars.rename(columns={mtcars.columns[0]: "model"})
mtcars.head()


In [ ]:
X = mtcars["wt"].values     # predictor: weight
y = mtcars["mpg"].values    # response: mpg

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train), " Test size:", len(X_test))


Fit the model on the training set using our `myLinReg` function:

In [ ]:
slope, intercept = myLinReg(X_train, y_train)
print(f"Fitted line:  mpg = {intercept:.4f} + ({slope:.4f}) * weight")


### 3. Predict mpg for a car with weight = 5.5 (i.e. 5500 lbs, since `wt` is in 1000 lbs units)

In [ ]:
weight_query = 5.5
predicted_mpg = myLinReg_predict(weight_query, slope, intercept)
print(f"Predicted mpg for a car with weight = {weight_query} (x1000 lbs): {predicted_mpg:.3f}")


### 4. Accuracy measures (RMSE, MAE) on the test set

In [ ]:
y_pred_test = myLinReg_predict(X_test, slope, intercept)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R^2 : {r2:.4f}")


### 5. Stochastic Gradient Descent (SGD) & Mini-Batch Gradient Descent
Implemented from scratch, with cost function (MSE) tracked and plotted over iterations.

In [ ]:
def compute_cost(x, y, slope, intercept):
    y_pred = intercept + slope * x
    return np.mean((y - y_pred) ** 2)


def sgd_linreg(x, y, lr=0.01, epochs=100):
    """Stochastic Gradient Descent: one random sample updates weights per step."""
    n = len(x)
    slope, intercept = 0.0, 0.0
    cost_history = []

    for epoch in range(epochs):
        indices = np.random.permutation(n)
        for i in indices:
            xi, yi = x[i], y[i]
            y_pred = intercept + slope * xi
            error = y_pred - yi

            grad_slope = 2 * error * xi
            grad_intercept = 2 * error

            slope -= lr * grad_slope
            intercept -= lr * grad_intercept

        cost_history.append(compute_cost(x, y, slope, intercept))

    return slope, intercept, cost_history


def mini_batch_gd_linreg(x, y, lr=0.01, epochs=100, batch_size=8):
    """Mini-Batch Gradient Descent: batch_size samples update weights per step."""
    n = len(x)
    slope, intercept = 0.0, 0.0
    cost_history = []

    for epoch in range(epochs):
        indices = np.random.permutation(n)
        x_shuffled, y_shuffled = x[indices], y[indices]

        for start in range(0, n, batch_size):
            end = start + batch_size
            xb, yb = x_shuffled[start:end], y_shuffled[start:end]

            y_pred = intercept + slope * xb
            error = y_pred - yb

            grad_slope = 2 * np.mean(error * xb)
            grad_intercept = 2 * np.mean(error)

            slope -= lr * grad_slope
            intercept -= lr * grad_intercept

        cost_history.append(compute_cost(x, y, slope, intercept))

    return slope, intercept, cost_history


# NOTE: features are standardized first -- gradient descent on raw 'wt'/'mpg'
# scales can diverge with a naive learning rate, so we scale, fit, then unscale.
x_mean, x_std = X_train.mean(), X_train.std()
y_mean, y_std = y_train.mean(), y_train.std()

X_train_scaled = (X_train - x_mean) / x_std
y_train_scaled = (y_train - y_mean) / y_std

sgd_slope, sgd_intercept, sgd_cost = sgd_linreg(X_train_scaled, y_train_scaled, lr=0.01, epochs=100)
mb_slope, mb_intercept, mb_cost = mini_batch_gd_linreg(X_train_scaled, y_train_scaled, lr=0.01, epochs=100, batch_size=8)

plt.figure(figsize=(8, 5))
plt.plot(sgd_cost, label="SGD")
plt.plot(mb_cost, label="Mini-Batch GD")
plt.xlabel("Epoch")
plt.ylabel("Cost (MSE, scaled)")
plt.title("Cost function over epochs")
plt.legend()
plt.show()

print(f"SGD final (scaled) slope/intercept: {sgd_slope:.4f}, {sgd_intercept:.4f}")
print(f"Mini-Batch GD final (scaled) slope/intercept: {mb_slope:.4f}, {mb_intercept:.4f}")


## Exercise 2

### 1. Load the Boston housing dataset and determine the best 5 features to predict `MEDV`
The classic `load_boston()` was removed from scikit-learn due to ethical concerns about one of its features, so we load it here from the original source (OpenML mirror).

In [ ]:
from sklearn.datasets import fetch_openml

boston = fetch_openml(name="boston", version=1, as_frame=True)
boston_df = boston.frame  # includes MEDV as target column
boston_df.head()


In [ ]:
# Correlation of every feature with MEDV -- rank by absolute correlation
correlations = boston_df.corr(numeric_only=True)["MEDV"].drop("MEDV")
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("Feature correlations with MEDV (sorted by strength):")
print(correlations_sorted)

best_5_features = correlations_sorted.index[:5].tolist()
print("\nBest 5 features:", best_5_features)


### 2. Multiple regression model using sklearn's `LinearRegression` with the best 3 features

In [ ]:
from sklearn.linear_model import LinearRegression

best_3_features = correlations_sorted.index[:3].tolist()
print("Best 3 features used for the model:", best_3_features)

X = boston_df[best_3_features]
y = boston_df["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print("Coefficients:", dict(zip(best_3_features, model.coef_)))
print("Intercept:", model.intercept_)


### 3. Accuracy of the model (RMSE, R²) on the 80/20 train/test split

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R^2 : {r2:.4f}")


---
**Notes**
- Question 3 in Exercise 1 asked for the prediction at `weight = 5.5`. Since the `mtcars` `wt` column is measured in units of 1000 lbs, this corresponds to a 5,500 lb car (quite heavy for the dataset's range) -- worth sanity-checking the result against the range of weights actually in the training data.
- Gradient descent was run on **standardized** data for numerical stability, then compared directly against the closed-form `myLinReg` solution in scaled terms. If your lab expects unscaled coefficients from GD, remove the standardization step and use a much smaller learning rate (e.g. `lr=0.0001`) to avoid divergence.
- If `fetch_openml` fails (no internet on the lab machine), download the Boston CSV once and load it locally with `pd.read_csv(...)` instead.